In [ ]:
!pip install datasets Pillow

In [ ]:
import math
import cv2
from collections import Counter

# 1. Mean
def mean(pixels):
    return sum(pixels) / len(pixels)

# 2. Median
def median(pixels):
    s = sorted(pixels)
    n = len(s)
    mid = n // 2
    if n % 2 == 0:
        return (s[mid - 1] + s[mid]) / 2
    return s[mid]

# 3. Variance
def variance(pixels):
    m = mean(pixels)
    return sum((x - m) ** 2 for x in pixels) / len(pixels)

# 4. Standard Deviation
def std_deviation(pixels):
    return math.sqrt(variance(pixels))

# 5. Skewness
def skewness(pixels):
    m   = mean(pixels)
    std = std_deviation(pixels)
    if std == 0:
        return 0.0
    return sum(((x - m) / std) ** 3 for x in pixels) / len(pixels)

# 6. Kurtosis  (excess kurtosis, normal = 0)
def kurtosis(pixels):
    m   = mean(pixels)
    std = std_deviation(pixels)
    if std == 0:
        return 0.0
    return sum(((x - m) / std) ** 4 for x in pixels) / len(pixels) - 3

# 7. Entropy  (Shannon entropy, base-2, in bits)
def entropy(pixels):
    n      = len(pixels)
    counts = Counter(pixels)
    result = 0.0
    for count in counts.values():
        p = count / n
        if p > 0:
            result -= p * math.log2(p)
    return result

# 8. Minimum Intensity
def min_intensity(pixels):
    minimum = pixels[0]
    for x in pixels:
        if x < minimum:
            minimum = x
    return minimum

# 9. Maximum Intensity
def max_intensity(pixels):
    maximum = pixels[0]
    for x in pixels:
        if x > maximum:
            maximum = x
    return maximum

# 10. Median Absolute Deviation (MAD)
def median_absolute_deviation(pixels):
    med       = median(pixels)
    deviations = [abs(x - med) for x in pixels]
    return median(deviations)

# 11. Local Contrast  (Michelson contrast)
def local_contrast(pixels):
    lo = min_intensity(pixels)
    hi = max_intensity(pixels)
    denom = hi + lo
    if denom == 0:
        return 0.0
    return (hi - lo) / denom

# 12. Local Probability  (frequency of a target value)
def local_probability(pixels, target_value):
    count = sum(1 for x in pixels if x == target_value)
    return count / len(pixels)

# 13. 25th Percentile
def percentile_25(pixels):
    return _percentile(pixels, 25)

# 14. 75th Percentile
def percentile_75(pixels):
    return _percentile(pixels, 75)

# shared helper
def _percentile(pixels, p):
    """Linear interpolation percentile (matches numpy default)."""
    s   = sorted(pixels)
    n   = len(s)
    idx = (p / 100) * (n - 1)          # fractional index
    lo  = int(idx)
    hi  = lo + 1
    frac = idx - lo
    if hi >= n:
        return float(s[-1])
    return s[lo] + frac * (s[hi] - s[lo])


#extract image feature and combine them in feature vector
def extract_image_features(image):
    img = np.array(image)

    if img is None or img.size == 0:
        print("Error: Empty or invalid image.")
        return None

    if img.ndim == 3:
        img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    pixels = img.flatten().tolist()
    most_common_val = Counter(pixels).most_common(1)[0][0]

    feature_vector = [
        mean(pixels),
        median(pixels),
        variance(pixels),
        std_deviation(pixels),
        skewness(pixels),
        kurtosis(pixels),
        entropy(pixels),
        min_intensity(pixels),
        max_intensity(pixels),
        median_absolute_deviation(pixels),
        local_contrast(pixels),
        local_probability(pixels, most_common_val),
        percentile_25(pixels),
        percentile_75(pixels),
    ]

    return feature_vector

In [ ]:
from datasets import load_dataset
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)

# Load MNIST
print("Loading MNIST dataset...")
dataset = load_dataset("mnist")

if 'train' not in dataset:
    raise ValueError("No train split found.")
if 'test' not in dataset:
    raise ValueError("No test split found.")

# Combine splits
all_data = list(dataset['train']) + list(dataset['test'])

# Extract Features
FEATURE_NAMES = [
    "Mean", "Median", "Variance", "Standard Deviation",
    "Skewness", "Kurtosis", "Entropy",
    "Minimum Intensity", "Maximum Intensity",
    "Median Absolute Deviation", "Local Contrast",
    "Local Probability", "25th Percentile", "75th Percentile"
]

features_list, labels = [], []

for item in all_data:
    fv = extract_image_features(item['image'])
    if fv is not None:
        features_list.append(fv)
        labels.append(item['label'])

if len(features_list) == 0:
    raise ValueError("No features extracted.")

# Feature DataFrame
feature_df = pd.DataFrame(features_list, columns=FEATURE_NAMES)

print("Feature Values (first 5 images):")
display(feature_df.head().style.set_properties(**{'background-color': 'white'}))

print("\nFeature Vector for first image:")
display(feature_df.iloc[[0]].style.set_properties(**{'background-color': 'white'}))

# Train / Test Split (70/30)
X = feature_df.values
y = pd.Series(labels)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

assert X_train.shape[1] == 14, "Feature vector must have 14 features."
assert X_train.shape[0] == y_train.shape[0], "Train/label mismatch."
assert X_test.shape[0]  == y_test.shape[0],  "Test/label mismatch."

print(f"\nTraining with {len(X_train)} samples, testing with {len(X_test)} samples.")

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

# Results
print("\n--- Classification Results ---")
print(f"Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision : {precision_score(y_test, y_pred, average='weighted', zero_division=0):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred, average='weighted', zero_division=0):.4f}")
print(f"F1-Score  : {f1_score(y_test, y_pred, average='weighted', zero_division=0):.4f}")

Loading MNIST dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

mnist/train-00000-of-00001.parquet:   0%|          | 0.00/15.6M [00:00<?, ?B/s]

mnist/test-00000-of-00001.parquet:   0%|          | 0.00/2.60M [00:00<?, ?B/s]

...

Feature Values (first 5 images):


,Mean,Median,Variance,Standard Deviation,Skewness,Kurtosis,Entropy,Minimum Intensity,Maximum Intensity,Median Absolute Deviation,Local Contrast,Local Probability,25th Percentile,75th Percentile
0,35.108418,0.000000,6343.935950,79.648829,2.057843,2.533948,1.828482,0,255,0.000000,1.000000,0.788265,0.000000,0.000000
1,39.661990,0.000000,7037.055392,83.887159,1.854180,1.697727,1.919594,0,255,0.000000,1.000000,0.775510,0.000000,0.000000
2,24.799745,0.000000,4300.703520,65.579749,2.566819,5.165020,1.455118,0,255,0.000000,1.000000,0.846939,0.000000,0.000000
3,21.855867,0.000000,4366.419277,66.078887,2.923698,6.877194,1.097717,0,255,0.000000,1.000000,0.877551,0.000000,0.000000
4,29.609694,0.000000,5531.092559,74.371315,2.362312,3.899435,1.609513,0,255,0.000000,1.000000,0.818878,0.000000,0.000000



Feature Vector for first image:


,Mean,Median,Variance,Standard Deviation,Skewness,Kurtosis,Entropy,Minimum Intensity,Maximum Intensity,Median Absolute Deviation,Local Contrast,Local Probability,25th Percentile,75th Percentile
0,35.108418,0.000000,6343.935950,79.648829,2.057843,2.533948,1.828482,0,255,0.000000,1.000000,0.788265,0.000000,0.000000



Training with 49000 samples, testing with 21000 samples.

--- Classification Results ---
Accuracy  : 0.3049
Precision : 0.2886
Recall    : 0.3049
F1-Score  : 0.2946
